## Configuration and Imports

In [ ]:
import json
import re
import os

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "../data/raw/medqa/medqa_base_eval.json"
OUTPUT_FAILURES = "failed_results_base_instruct_8b.json"


## Extract EDA Logic

In [ ]:
def extract_letter(text):
    """
    Extracts a single uppercase letter (A-E) from a string.
    Handles formats like "A", "A)", "(A)", or "The answer is A".
    """
    if not text:
        return None

    # Look for a pattern like "Answer is A" or "#### A" or just "A" at the start/end
    # This regex looks for a lone letter A-E, potentially near punctuation
    match = re.search(r"(?i)\b([A-E])\b", str(text))
    if match:
        return match.group(1).upper()

    # Fallback: just grab the first A-E character found if no word boundaries match
    match = re.search(r"[A-E]", str(text).upper())
    if match:
        return match.group(0)

    return None


def run_evaluation():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ File not found: {INPUT_FILE}")
        return

    with open(INPUT_FILE, "r") as f:
        data = json.load(f)

    correct_count = 0
    total = len(data)
    failed_items = []

    print(f"{'ID':<5} | {'Truth':<6} | {'Pred':<6} | {'Status'}")
    print("-" * 40)

    for item in data:
        # Get ground truth
        gt = str(item.get("ground_truth", "")).strip().upper()

        # Extract the letter from raw_output
        raw = item.get("raw_output", "")
        extracted = extract_letter(raw)

        # Comparison logic
        is_correct = extracted == gt
        item["is_correct"] = is_correct
        item["predicted_letter"] = extracted  # Update with cleaned letter

        if is_correct:
            correct_count += 1
            status = "✅"
        else:
            status = "❌"
            failed_items.append(item)

        print(f"{item.get('id', '?'):<5} | {gt:<6} | {str(extracted):<6} | {status}")

    # --- STATS ---
    accuracy = (correct_count / total) * 100 if total > 0 else 0
    print("-" * 40)
    print(f"📊 SUMMARY:")
    print(f"Total:    {total}")
    print(f"Correct:  {correct_count}")
    print(f"Accuracy: {accuracy:.2f}%")

    # --- SAVE FAILURES ---
    if failed_items:
        with open(OUTPUT_FAILURES, "w") as f:
            json.dump(failed_items, f, indent=4)
        print(f"\n💾 Saved {len(failed_items)} incorrect cases to {OUTPUT_FAILURES}")


## Execution

In [ ]:
if __name__ == "__main__":
    run_evaluation()
